### Categorical Training Notebook

This notebook contains code to train classification models on curated price data (specifically SMA and RSI).

In [ ]:
# Imports

import tensorflow as tf
import numpy as np

In [ ]:
# Categorization function for labels
def categorize_labels(label: float) -> float:
    if label < -1:
        return 0.0
    if label < 0:
        return 1.0
    if label < 1:
        return 2.0
    if label < 1.5:
        return 3.0
    
    return 4.0

In [ ]:
# Load data and shuffle it
data = np.loadtxt('stock_sma_3_3_curated.csv', delimiter=',')
data = data.T
np.random.shuffle(data)
data = data.T

In [ ]:
# Load X and categorize and one hot encode Y labels
X = data[0:504,:]
raw_Y = np.array(data[-1:,:])
Y = (tf.keras.utils.to_categorical(np.vectorize(categorize_labels)(raw_Y), 2)).T

In [ ]:
Y.shape = (Y.shape[0], Y.shape[1])
print(X.shape)
print(Y.shape)

In [ ]:
print("Very Bear:", np.count_nonzero(Y[0]))
print("Bear:", np.count_nonzero(Y[1]))
print("Neutral:", np.count_nonzero(Y[2]))
print("Bull:", np.count_nonzero(Y[3]))
print("Very Bull:", np.count_nonzero(Y[4]))

In [ ]:
# Initialize the classification model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8), input_shape=(504,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(5, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Initialize training and validation data
train_size = int(data.shape[1] * 0.8)

X_train = X[:,:train_size].T
Y_train = Y[:,:train_size].T

X_val = X[:,train_size:].T
Y_val = Y[:,train_size:].T

print(X_train.shape)
print(Y_train.shape)
print(X_val.shape)
print(Y_val.shape)

In [ ]:
# Train the model
model.fit(X_train, Y_train,
                  validation_data=(X_val,Y_val),
                  batch_size=64, 
                  epochs=30,
                  verbose=1,
                  shuffle=True)

In [ ]:
import matplotlib.pyplot as plt

# Function for plotting accuracy and loss for the model history
def plot_acc_and_loss(history, title):

    #Plot the validation and training accuracy
    plt.title("Model and Validation Accuracy for " + title)
    xs = list(range(1, len(history.history['accuracy']) + 1))
    plt.plot(xs, history.history['accuracy'], label="Model Accuracy", color="Red")
    plt.plot(xs, history.history['val_accuracy'], label="Validation Accuracy", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    #Plot the validation and training loss
    plt.title("Model and Validation Loss for " + title)
    xs = list(range(1, len(history.history['val_loss']) + 1))
    plt.plot(xs, history.history['loss'], label="Model loss", color="Red")
    plt.plot(xs, history.history['val_loss'], label="Validation loss", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Cost")
    plt.legend()
    plt.show()

In [ ]:
# Display history results for the initial regression model
plot_acc_and_loss(model.history, "Categorized SMA")

In [ ]:
# Print some predictions for analysis
print("Prediction:", model(X_val[5:10]))
print("Label:", Y_val[5:10])

In [ ]:
# Not covered in report, a thin model for classification
modelThin = tf.keras.Sequential([
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8), input_shape=(504,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(2, activation='softmax'),
])

modelThin.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

modelThin.summary()

In [ ]:
# Train the thin model
modelThin.fit(X_train, Y_train,
                  validation_data=(X_val,Y_val),
                  batch_size=64, 
                  epochs=30,
                  verbose=1,
                  shuffle=True)

In [ ]:
# Display results for the thin model
plot_acc_and_loss(model.history, "Thin Categorized SMA")